#### An AI agent to automate research : it will take a question, search the web, read the top results, and give the most relevant passages back along with a short summary

##### The core idea isn’t just to find pages with the right keywords, but to find passages with the right meaning, For his, use vector embeddings - a coordinate of meaning in a high-dumensional space. 

##### The sentence-transformers library provides models that are expert at tuning any piece of text into a list of numbers - a vector, that represents its location in that "meaningful space"

In [1]:
# All needed imports 

# Data Science and NLP Libraries
from bs4 import BeautifulSoup
from ddgs import DDGS  # DDGS library allows you to perform web searches directly from Python
from huggingface_hub import InferenceClient
import numpy as np
from sentence_transformers import SentenceTransformer

# Other Libraries
import re
import os
import time
import urllib.parse
import requests

/root/opt/ra-i-g/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Testing the models, cause we need a free tier model
client = InferenceClient(token=os.getenv("HF_TOKEN"))

models_to_test = [
    "meta-llama/Meta-Llama-3-8B-Instruct",
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "mistralai/Mixtral-8x7B-Instruct-v0.1",
    "Qwen/Qwen2.5-7B-Instruct",
    "microsoft/Phi-3-mini-4k-instruct",
    "google/gemma-2-9b-it",
    "HuggingFaceH4/zephyr-7b-beta",
]

for m in models_to_test:
    try:
        result = client.chat_completion(
            messages=[{"role": "user", "content": "Say hello"}],
            model=m,
            max_tokens=10
        )
        print(f"{m}: WORKS - {result.choices[0].message.content}")
    except Exception as e:
        print(f"{m}: FAILED - {str(e)[:80]}")

meta-llama/Meta-Llama-3-8B-Instruct: FAILED - (Request ID: Root=1-6a590758-54ac53c14a12523352a5569c;c500818d-95a7-4f41-884c-a9
meta-llama/Meta-Llama-3.1-8B-Instruct: FAILED - (Request ID: Root=1-6a590758-4a8082f879fd3547786f8389;94dc4e1e-f541-4956-8197-c7
mistralai/Mistral-7B-Instruct-v0.3: FAILED - (Request ID: Root=1-6a590758-59c6ae013b50b0e51a029a0d;958db577-30b9-49f1-8b1e-b2
mistralai/Mixtral-8x7B-Instruct-v0.1: FAILED - (Request ID: Root=1-6a590758-271fa2c73b5623d055810eb9;16275397-8a47-45d8-a3ef-cf
Qwen/Qwen2.5-7B-Instruct: WORKS - Hello! How can I assist you today?
microsoft/Phi-3-mini-4k-instruct: FAILED - (Request ID: Root=1-6a590759-2d356da10b3194673154a1fc;5d952288-a8d5-4575-b512-30
google/gemma-2-9b-it: FAILED - (Request ID: Root=1-6a590759-22f309c92b09d75e64c2b856;13d94c60-e442-40dc-813c-03
HuggingFaceH4/zephyr-7b-beta: FAILED - (Request ID: Root=1-6a590759-79ed00f0538a834a435e4e10;6cde578b-5c2f-45c0-a85f-01


In [8]:
# Configuration and Settings
SEARCH_RESULTS = 6        # How many URLs to check
PASSAGES_PER_PAGE = 4     # How many passages to pull from each URL
TOP_PASSAGES = 5          # How many relevant passages to use for the summary
SUMMARY_SENTENCES = 3     # How many sentences in the final summary
TIMEOUT = 8               # How long to wait for a webpage to load
LLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2" # Fast, high-quality model

llm_client = InferenceClient(token=os.getenv("HF_TOKEN"))

In [9]:
# Web Search with DuckDuckGo
def unwrap_ddg(url):
    """Extracts real URL from DuckDuckGo redirect wrapper."""
    try:
        parsed = urllib.parse.urlparse(url)
        if "duckduckgo.com" in parsed.netloc:
            qs = urllib.parse.parse_qs(parsed.query)
            uddg = qs.get("uddg")
            if uddg:
                return urllib.parse.unquote(uddg[0])
    except Exception:
        pass
    return url

def search_web(query, max_results=SEARCH_RESULTS):
    """Searches the web and return a list of URLs."""
    urls = []
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=max_results):
            url = r.get("href") or r.get("url")
            if not url:
                continue
            url = unwrap_ddg(url) # Clean up DDG redirect links
            urls.append(url)
    return urls

In [10]:
# Fetch and Clean the Web Pages with python libraries: requests and BeautifulSoup

def fetch_text(url, timeout=TIMEOUT):
    """Fetches and cleans text content from a URL."""
    headers = {"User-Agent": "Mozilla/5.0 (research-agent)"}
    try:
        r = requests.get(url, timeout=timeout, headers=headers, allow_redirects=True)
        if r.status_code != 200:
            return ""
        ct = r.headers.get("content-type", "")
        if "html" not in ct.lower(): # Skipping non-HTML content
            return ""
        
        soup = BeautifulSoup(r.text, "html.parser")
        
        # Removing all annoying tags
        for tag in soup(["script", "style", "noscript", "header", "footer", "svg", "iframe", "nav", "aside"]):
            tag.extract()
            
        # Get all paragraph text
        paragraphs = [p.get_text(" ", strip=True) for p in soup.find_all("p")]
        text = " ".join([p for p in paragraphs if p])
        
        if text.strip():
            # Cleaning up whitespace
            return re.sub(r"\s+", " ", text).strip()
            
        # --- Fallback logic if <p> tags fail ---
        meta = soup.find("meta", attrs={"name": "description"}) or soup.find("meta", attrs={"property": "og:description"})
        if meta and meta.get("content"):
            return meta["content"].strip()
        if soup.title and soup.title.string:
            return soup.title.string.strip()
            
    except Exception:
        return "" # Fails silently
    return ""

In [11]:
# Chunking, embedding, and ranking the text passages - breaking long articles into smaller passges and embedding them using SentenceTransformers, then ranking them based on their relevance to the query.
def chunk_passages(text, max_words=120):
    """Splits long text into smaller passages."""
    words = text.split()
    if not words:
        return []
    chunks = []
    i = 0
    while i < len(words):
        chunk = words[i : i + max_words]
        chunks.append(" ".join(chunk))
        i += max_words
    return chunks

def split_sentences(text):
    """Splits text into sentences."""
    parts = re.split(r'(?<=[.!?])\s+', text)
    return [p.strip() for p in parts if p.strip()]

def cosine(a, b):
    """Computes cosine similarity between two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10)


In [12]:
# Research Agent
class ResearchAgent:
    def __init__(self, embed_model=EMBEDDING_MODEL):
        print(f"Loading embedder: {embed_model}...")
        self.embedder = SentenceTransformer(embed_model)

    def run(self, query, use_llm_summary=False):
        start = time.time()
        
        # Starting the search
        urls = search_web(query)
        print(f"Found {len(urls)} urls.")
        
        # Fetch & Chunk
        docs = []
        for u in urls:
            txt = fetch_text(u)
            if not txt:
                continue
            chunks = chunk_passages(txt, max_words=120)
            for c in chunks[:PASSAGES_PER_PAGE]:
                docs.append({"url": u, "passage": c})
        
        if not docs:
            print("No documents fetched.")
            return {"query": query, "passages": [], "summary": ""}
        
        # Embedding
        print(f"Embedding {len(docs)} passages...")
        texts = [d["passage"] for d in docs]
        emb_texts = self.embedder.encode(texts, convert_to_numpy=True, show_progress_bar=False)
        q_emb = self.embedder.encode([query], convert_to_numpy=True)[0]

        # Ranking by similarity
        sims = [cosine(e, q_emb) for e in emb_texts]
        top_idx = np.argsort(sims)[::-1][:TOP_PASSAGES]
        top_passages = [{
            "url": docs[i]["url"],
            "passage": docs[i]["passage"],
            "score": float(sims[i])
        } for i in top_idx]

        # Summarization
        if use_llm_summary:
            summary = self._llm_summary(query, top_passages)
        else:
            summary = self._extractive_summary(query, q_emb, top_passages)

        elapsed = time.time() - start
        return {
            "query": query,
            "passages": top_passages,
            "summary": summary,
            "time": elapsed
        }
    
    def _extractive_summary(self, query, q_emb, top_passages):
        """generates summary using extractive method (no LLM needed)."""
        sentences = []
        for tp in top_passages:
            for s in split_sentences(tp["passage"]):
                sentences.append({"sent": s, "url": tp["url"]})

        if not sentences:
            return "No summary could be generated."

        sent_texts = [s["sent"] for s in sentences]
        sent_embs = self.embedder.encode(sent_texts, convert_to_numpy=True, show_progress_bar=False)
        sent_sims = [cosine(e, q_emb) for e in sent_embs]

        top_sent_idx = np.argsort(sent_sims)[::-1][:SUMMARY_SENTENCES]
        chosen = [sentences[idx] for idx in top_sent_idx]

        seen = set()
        lines = []
        for s in chosen:
            key = s["sent"].lower()[:80]
            if key in seen:
                continue
            seen.add(key)
            lines.append(f"{s['sent']} (Source: {s['url']})")
        return " ".join(lines)

    def _llm_summary(self, query, top_passages):
        """Generates summary using remote LLM."""
        context = "\n\n".join(
            f"[Source: {p['url']}]\n{p['passage']}" for p in top_passages
        )
        result = llm_client.chat_completion(
            messages=[{
                "role": "user",
                    "content": f"""Based on the research passages below, provide a clear, 
                    concise summary that answers the question. Cite the sources.

                    Question: {query}

                    Research Passages:
                    {context}

                    Summary:"""
                }],
                model=LLM_MODEL,
                max_tokens=300
            )
        return result.choices[0].message.content.strip()


In [13]:
agent = ResearchAgent()

q = "What causes the long heat waves in Europe and how they originate?"
print(f"\nResearching: {q}\n")
out = agent.run(q, use_llm_summary=True)

print("\nTop passages:")
for p in out["passages"]:
    print(f"  score {p['score']:.3f} | {p['url']}")
    print(f"  {p['passage'][:150]}...\n")

print("--- Summary ---")
print(out["summary"])
print("---------------")
print(f"\nDone in {out['time']:.1f}s")

Loading embedder: sentence-transformers/all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14291.35it/s]



Researching: What causes the long heat waves in Europe and how they originate?

Found 6 urls.
Embedding 5 passages...

Top passages:
  score 0.563 | https://sustainablecarbon.com/what-causes-heat-waves-in-europe/
  Summer is at its peak in Europe: temperatures reached 38,6ºC at Coschen, near the border with Poland. In Germany, the asphalt from a road in the cente...

  score 0.525 | https://www.shankariasparliament.com/current-affairs/europe-heatwave
  creates and traps a heat dome at the surface Aspect Heat Dome (Cause) Heatwave (Effect) Definition A strong, persistent high‑pressure system that trap...

  score 0.508 | https://www.shankariasparliament.com/current-affairs/europe-heatwave
  Click here to download | June 2026 - Prelim Bits | Mains | UPSC PRELIMS 2026 (GS - PAPER 1) Shankar IAS Parliament Reflections | Mainstorming 2026 - S...

  score 0.467 | https://sustainablecarbon.com/what-causes-heat-waves-in-europe/
  are not uncommon, but are being amplified by the rise in temper